# VCD Table 1 Reproduction (LLaVA-1.5, POPE random/popular)
This notebook runs Regular decoding vs VCD and reports Accuracy, Precision, Recall, and F1.

In [7]:
import torch
print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected')

2.9.0+cu128
CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB


In [11]:
# Public repo URL
REPO_URL = 'https://github.com/maxheef/ECE209_Project1.git'
PROJECT_DIR = '/content/VCD_project'
import os
os.environ['REPO_URL'] = REPO_URL
os.environ['PROJECT_DIR'] = PROJECT_DIR
print('Using repo:', REPO_URL)

Using repo: https://github.com/maxheef/ECE209_Project1.git


In [12]:
%%bash
set -e
cd /content
if [ ! -d VCD_project/.git ]; then
  rm -rf VCD_project
  GIT_TERMINAL_PROMPT=0 git clone --depth 1 "$REPO_URL" VCD_project
fi

# Detect project root that contains both VCD/ and myCode/
if [ -d /content/VCD_project/VCD ] && [ -d /content/VCD_project/myCode ]; then
  PROJ=/content/VCD_project
elif [ -d /content/VCD_project/VCD_project/VCD ] && [ -d /content/VCD_project/VCD_project/myCode ]; then
  PROJ=/content/VCD_project/VCD_project
else
  echo 'Could not find expected project layout under /content/VCD_project'
  find /content/VCD_project -maxdepth 3 -type d | sed -n '1,120p'
  exit 1
fi
echo "Using project root: $PROJ"
echo "$PROJ" > /tmp/vcd_project_root.txt
cd "$PROJ/VCD"
python3 -m pip install --upgrade pip
pip install -r requirements.txt

Could not find expected project layout under /content/VCD_project
/content/VCD_project
/content/VCD_project/.git
/content/VCD_project/.git/refs
/content/VCD_project/.git/refs/tags
/content/VCD_project/.git/refs/heads
/content/VCD_project/.git/objects
/content/VCD_project/.git/objects/pack
/content/VCD_project/.git/objects/info
/content/VCD_project/.git/hooks
/content/VCD_project/.git/info
/content/VCD_project/.git/branches


CalledProcessError: Command 'b'set -e\ncd /content\nif [ ! -d VCD_project/.git ]; then\n  rm -rf VCD_project\n  GIT_TERMINAL_PROMPT=0 git clone --depth 1 "$REPO_URL" VCD_project\nfi\n\n# Detect project root that contains both VCD/ and myCode/\nif [ -d /content/VCD_project/VCD ] && [ -d /content/VCD_project/myCode ]; then\n  PROJ=/content/VCD_project\nelif [ -d /content/VCD_project/VCD_project/VCD ] && [ -d /content/VCD_project/VCD_project/myCode ]; then\n  PROJ=/content/VCD_project/VCD_project\nelse\n  echo \'Could not find expected project layout under /content/VCD_project\'\n  find /content/VCD_project -maxdepth 3 -type d | sed -n \'1,120p\'\n  exit 1\nfi\necho "Using project root: $PROJ"\necho "$PROJ" > /tmp/vcd_project_root.txt\ncd "$PROJ/VCD"\npython3 -m pip install --upgrade pip\npip install -r requirements.txt\n'' returned non-zero exit status 1.

In [ ]:
%%bash
set -e
mkdir -p /content/datasets/coco
cd /content/datasets/coco
if [ ! -d val2014 ]; then
  wget -q http://images.cocodataset.org/zips/val2014.zip
  unzip -q val2014.zip
fi
ls -la /content/datasets/coco/val2014 | head

In [ ]:
import os
MODEL_PATH = 'liuhaotian/llava-v1.5-7b'  # or local model path
COCO_VAL_DIR = '/content/datasets/coco/val2014'
SEED = 55
CD_ALPHA = 1.0
CD_BETA = 0.2
NOISE_STEP = 500

PROJECT_DIR = open('/tmp/vcd_project_root.txt').read().strip() if os.path.exists('/tmp/vcd_project_root.txt') else '/content/VCD_project'
os.environ['MODEL_PATH'] = str(MODEL_PATH)
os.environ['COCO_VAL_DIR'] = str(COCO_VAL_DIR)
os.environ['SEED'] = str(SEED)
os.environ['CD_ALPHA'] = str(CD_ALPHA)
os.environ['CD_BETA'] = str(CD_BETA)
os.environ['NOISE_STEP'] = str(NOISE_STEP)
os.environ['PROJECT_DIR'] = PROJECT_DIR
print('PROJECT_DIR =', PROJECT_DIR)

In [ ]:
%%bash
set -e
PROJECT_DIR=${PROJECT_DIR:-/content/VCD_project}
cd "$PROJECT_DIR"
PYTHON_BIN=python3 \
CD_ALPHA=$CD_ALPHA CD_BETA=$CD_BETA NOISE_STEP=$NOISE_STEP \
bash myCode/run_table1_repro.sh \
  "$MODEL_PATH" \
  "$COCO_VAL_DIR" \
  "$SEED"

In [ ]:
%%bash
set -e
PROJECT_DIR=${PROJECT_DIR:-/content/VCD_project}
cd "$PROJECT_DIR"
ls -la myCode/output
echo '\n--- random regular ---'
cat myCode/output/metrics_random_regular_seed55.json
echo '\n--- random vcd ---'
cat myCode/output/metrics_random_vcd_seed55.json
echo '\n--- popular regular ---'
cat myCode/output/metrics_popular_regular_seed55.json
echo '\n--- popular vcd ---'
cat myCode/output/metrics_popular_vcd_seed55.json
echo '\n--- table ---'
python3 myCode/table1_from_outputs.py --outputs myCode/output --gt-root VCD/experiments/data/POPE/coco --seed 55